In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

In [4]:
df = pd.read_csv('train_final.csv')

In [5]:
df.head()

,Unnamed: 0,user_id,ts,gate_id
0,0,18,2022-07-29 09:08:54,7
1,1,18,2022-07-29 09:09:54,9
2,2,18,2022-07-29 09:09:54,9
3,3,18,2022-07-29 09:10:06,5
4,4,18,2022-07-29 09:10:08,5


### --- Feature Engineering ---

In [6]:
# --- 1. Feature Engineering (Полный набор данных) ---
df['ts'] = pd.to_datetime(df['ts'])
df = df.sort_values('ts')

# Создание базовых временных признаков
df['hour'] = df['ts'].dt.hour
df['min'] = df['ts'].dt.minute
df['day_of_week'] = df['ts'].dt.dayofweek
df['day_of_month'] = df['ts'].dt.day

# Cyclical Features (для Hour и DOW)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Признак взаимодействия (Gate + DOW)
df['gate_dow'] = df['gate_id'].astype(str) + '_' + df['day_of_week'].astype(str)

# Lag Feature (Числовой)
df['time_diff'] = df['ts'].diff().dt.total_seconds().fillna(0)
# Очистка
df = df.drop(columns=['ts'])

# --- 2. Target Encoding ---
# Кодируем user_id в числа (требуется для scikit-learn)
le = LabelEncoder()
df['user_id_encoded'] = le.fit_transform(df['user_id'])

# --- 3. Modeling Setup ---
X = df.drop(columns=['user_id', 'user_id_encoded'])
Y = df['user_id_encoded']

In [7]:
df.columns.unique()

Index(['Unnamed: 0', 'user_id', 'gate_id', 'hour', 'min', 'day_of_week',
       'day_of_month', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
       'gate_dow', 'time_diff', 'user_id_encoded'],
      dtype='object')

### Model

In [8]:
# Разделение данных (используем весь набор, без фильтрации)
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# --- 4. Preprocessing Pipeline (ColumnTransformer) ---
numerical_features = ['time_diff', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'min', 'day_of_month']
categorical_features = ['gate_id', 'day_of_week', 'hour', 'gate_dow']

# Создание преобразователя
preprocessor = ColumnTransformer(
    transformers=[
        # 1. Standard Scaler для числовых признаков (для регуляризации)
        ('num', StandardScaler(), numerical_features),
        # 2. OneHotEncoder для категориальных признаков
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features)
    ],
    remainder='passthrough' # Оставить остальные признаки как есть (на данный момент их нет)
)

# Преобразование данных
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

In [9]:
# --- 4. Hyperparameter Tuning (Grid Search) ---

# Определение моделей и сеток параметров
pipeline_models = {
'KNN': (KNeighborsClassifier(n_jobs=-1), {
        'n_neighbors': [2, 3, 4, 5],  # Количество соседей
        'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'weights': ['uniform', 'distance'], # Взвешивание
        'p': [1, 2] # Метрика: 1=Manhattan, 2=Euclidean
    }),
    
    'RandomForest': (RandomForestClassifier(random_state=42, n_jobs=-1), {
        'n_estimators': [100, 150], # Количество деревьев
        'max_depth': [15, 20],   # Максимальная глубина
        'min_samples_split': [5, 8] # Минимальное количество выборок для разделения
    })
}

best_results = {}
best_score = 0
best_model_name = ""
best_model = None

# Запускаем Grid Search для каждой модели
for name, (model, params) in pipeline_models.items():
    print(f"\n--- Running GridSearchCV for {name} ---")
    
    # Grid Search с кросс-валидацией (CV=3) и метрикой f1_weighted
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=params,
        scoring='f1_weighted', # Целевая метрика для дисбаланса
        cv=3,
        verbose=1,
        n_jobs=-1
    )
    
    # Обучение
    grid_search.fit(X_train_processed, Y_train)
    
    # Получение лучшего результата
    current_score = grid_search.best_score_
    
    # Сохранение результатов
    best_results[name] = {
        'Best Score (CV)': current_score,
        'Best Params': grid_search.best_params_
    }
    
    # Проверка, является ли текущая модель лучшей
    if current_score > best_score:
        best_score = current_score
        best_model_name = name
        best_model = grid_search.best_estimator_
        
    print(f"Best CV Score (Weighted F1): {current_score:.4f}")
    print(f"Best Parameters: {grid_search.best_params_}")


# --- 5. Финальная Оценка Лучшей Модели ---

print("\n\n--- Финальная Оценка Лучшей Модели ---")
print(f"Лучшая модель по результатам Grid Search: **{best_model_name}**")

# Предсказание на валидационном наборе
Y_preds = best_model.predict(X_val_processed)
acc = accuracy_score(Y_val, Y_preds)

# Декодирование меток обратно в исходные ID пользователей
Y_val_original = le.inverse_transform(Y_val)
Y_preds_original = le.inverse_transform(Y_preds)

print(f"\nValidation Accuracy (Best Model): {acc:.4f}")
print("Classification Report:")
print(classification_report(Y_val_original, Y_preds_original, zero_division=0))

print("\nHyperparameter Results Summary:")
for name, res in best_results.items():
    print(f"**{name}**: Score={res['Best Score (CV)']:.4f}, Params={res['Best Params']}")


--- Running GridSearchCV for KNN ---
Fitting 3 folds for each of 64 candidates, totalling 192 fits
Best CV Score (Weighted F1): 0.5846
Best Parameters: {'algorithm': 'auto', 'n_neighbors': 2, 'p': 1, 'weights': 'distance'}

--- Running GridSearchCV for RandomForest ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CV Score (Weighted F1): 0.2722
Best Parameters: {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 150}


--- Финальная Оценка Лучшей Модели ---
Лучшая модель по результатам Grid Search: **KNN**

Validation Accuracy (Best Model): 0.6936
Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.63      0.64       250
           1       0.76      0.75      0.75       260
           2       0.83      0.62      0.71         8
           3       0.78      0.75      0.76       198
           5       1.00      0.50      0.67         2
           6       0.67      0.68      0.67       403
           7      

In [20]:
test_final = pd.read_csv('test_final.csv')

test_final['ts'] = pd.to_datetime(test_final['ts'])

test_final['hour'] = test_final['ts'].dt.hour
test_final['minute'] = test_final['ts'].dt.minute
test_final['day_of_week'] = test_final['ts'].dt.dayofweek
test_final['day_of_month'] = test_final['ts'].dt.day
test_final['month'] = test_final['ts'].dt.month
test_final['hour_sin'] = np.sin(2 * np.pi * test_final['hour']/24)
test_final['hour_cos'] = np.cos(2 * np.pi * test_final['hour']/24)
test_final['day_sin'] = np.sin(2 * np.pi * test_final['day_of_week']/7)
test_final['day_cos'] = np.cos(2 * np.pi * test_final['day_of_week']/7)
test_final['is_weekend'] = test_final['day_of_week'].isin([5, 6]).astype(int)
test_final['is_morning'] = ((test_final['hour'] >= 8) & (test_final['hour'] <= 11)).astype(int)
test_final['is_afternoon'] = ((test_final['hour'] >= 12) & (test_final['hour'] <= 17)).astype(int)

# Удаляем временные метки и лишние колонки
X_test_final = test_final.drop(['ts', ''], axis=1, errors='ignore')

predictions = best_model.predict(X_test_final)

# Создаем DataFrame для записи
submission = pd.DataFrame({
    'user_word': range(len(predictions)),
    'preds': predictions
})

# Сохраняем в файл
submission.to_csv('answer_final.csv', index=False)

print(f"Submission saved to csv, shape: {submission.shape}")

ValueError: could not convert string to float: 'gini'

In [22]:
X_train_processed.shape

(30014, 142)

In [25]:
df_test=pd.read_csv('test_final.csv')
# --- 1. Feature Engineering  ---
df_test['ts'] = pd.to_datetime(df_test['ts'])
df_test = df_test.sort_values('ts')

df_test['hour'] = df_test['ts'].dt.hour
df_test['min'] = df_test['ts'].dt.minute
df_test['day_of_week'] = df_test['ts'].dt.dayofweek
df_test['day_of_month'] = df_test['ts'].dt.day

df_test['hour_sin'] = np.sin(2 * np.pi * df_test['hour'] / 24)
df_test['hour_cos'] = np.cos(2 * np.pi * df_test['hour'] / 24)
df_test['dow_sin'] = np.sin(2 * np.pi * df_test['day_of_week'] / 7)
df_test['dow_cos'] = np.cos(2 * np.pi * df_test['day_of_week'] / 7)

# Признак взаимодействия (Gate + DOW)
df_test['gate_dow'] = df_test['gate_id'].astype(str) + '_' + df_test['day_of_week'].astype(str)
# Lag Feature (Числовой)
df_test['time_diff'] = df_test['ts'].diff().dt.total_seconds().fillna(0)
# Очистка
df_test = df_test.drop(columns=['ts'])

# --- 2. Target Encoding ---
# Кодируем user_id в числа (требуется для scikit-learn)
#df_test['user_id_encoded'] = le.fit_transform(df_test['user_id'])

# --- 3. Modeling Setup ---
#X = df_test.drop(columns=['user_id', 'user_id_encoded'])
#Y = df['user_id_encoded']
df_test.columns.unique()

Index(['Unnamed: 0', 'gate_id', 'user_word', 'hour', 'min', 'day_of_week',
       'day_of_month', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
       'gate_dow', 'time_diff'],
      dtype='object')

In [29]:
df_word=df_test['user_word']
df_test=df_test.drop(columns=['user_word'])

In [24]:
##numerical_features = ['time_diff', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'min', 'day_of_month']
#categorical_features = ['gate_id', 'day_of_week', 'hour', 'gate_dow']
#'user_id', 'gate_id', 'hour', 'min', 'day_of_week',        'day_of_month', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',       'gate_dow', 'time_diff', 'user_id_encoded'],
df_test.head()


,Unnamed: 0,gate_id,user_word,hour,min,day_of_week,day_of_month,dow_sin,dow_cos,gate_dow,time_diff
0,37518,9,gini,8,21,1,3,0.781831,0.62349,9_1,0.0
1,37519,9,gini,8,21,1,3,0.781831,0.62349,9_1,0.0
2,37520,5,gini,8,21,1,3,0.781831,0.62349,5_1,18.0
3,37521,5,gini,8,21,1,3,0.781831,0.62349,5_1,1.0
4,37522,10,gini,8,21,1,3,0.781831,0.62349,10_1,20.0


In [40]:
df_test.shape

(7125, 12)

In [30]:
df_test_processed = preprocessor.transform(df_test)
predictions_test = best_model.predict(df_test_processed)


In [36]:
df_test_processed.shape

(7125, 142)

In [31]:
submission_test = pd.DataFrame({
    'user_word': df_word,
    'preds': predictions_test
})

In [34]:
submission_test.head(15)

,user_word,preds
0,gini,6
1,gini,6
2,gini,6
3,gini,6
4,gini,6
5,epsilon,6
6,epsilon,6
7,epsilon,6
8,epsilon,6
9,epsilon,6


In [32]:
submission_test.to_csv('answer_final.csv', index=False)